## Implementing Guardrails

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import os
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

In [3]:
from langchain_groq import ChatGroq
model = ChatGroq(model = "llama-3.1-8b-instant", api_key = GROQ_API_KEY)

d:\Namit Kumar\Learning\Agentic AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 1. Deterministic Guardrail Example

In [4]:
def deterministic_guardrail(text: str)  -> bool:
    """Return True if the content is unsafe"""
    banned_keywords = ["hack", "exploit", "malware", "bomb"]

    return any(kw in text.lower() for kw in banned_keywords)


test_inputs = [
    "How to hack a database",
    "What is Capital of france",
    "Explain how malware spreads"
]


for input in test_inputs:
    blocked = deterministic_guardrail(input)
    status = "Blocked" if blocked else "Allowed"
    print(f"{status} : {input}")

Blocked : How to hack a database
Allowed : What is Capital of france
Blocked : Explain how malware spreads


### 2. Model based Guardrail Example

In [5]:
from langchain_groq import ChatGroq

def model_based_guardrail(text: str) -> str:
    """Uses a LLM to evaluate whether user input is safe or not, Returns SAFE or UNSAFE"""

    # Here we can also use Pydantic or TypedDict also
    prompt = f"""
    Is the following user text is safe to process ?

    Reply with only "SAFE" or "UNSAFE"
    
    Input : {text}
    
    """

    result = model.invoke(prompt)

    return result.content.strip()

for input in test_inputs:
    verdict = model_based_guardrail(input)
    status = "Blocked" if "UNSAFE" in verdict else "Allowed"
    print(f"{status} : {input}")

Blocked : How to hack a database
Allowed : What is Capital of france
Allowed : Explain how malware spreads


### 3. PII Middleware

It is applied before input goes to the Agent or before the Agent is called

In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware

# Dummy tool
def customer_lookup(query: str) -> str:
    """Lookup for customer information"""

    return f"Customer information found for query : {query}"


agent = create_agent(
    model = model,
    tools = [customer_lookup],
    middleware = [
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True # Basically apply to the inputs
        ),
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True
        ),
        PIIMiddleware(
            "api_key",
            detector = r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True # Basically apply to the inputs
        ),
    ]
)

In [15]:
result = agent.invoke({"messages" : "My email is john.doe@example.com and my credit card is 5555 5555 5555 4444. Can you help me ?"})


print(result["messages"][-1].content)

I can't assist with that request. If you'd like to look up customer information, I can help with that.


In [16]:
result

{'messages': [HumanMessage(content='My email is [REDACTED_EMAIL] and my credit card is **** **** **** 4444. Can you help me ?', additional_kwargs={}, response_metadata={}, id='2efd2b7e-7a7a-4340-a9d2-d1effd40c3e7'),
  AIMessage(content="I can't assist with that request. If you'd like to look up customer information, I can help with that.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 235, 'total_tokens': 260, 'completion_time': 0.075941582, 'completion_tokens_details': None, 'prompt_time': 0.014493005, 'prompt_tokens_details': None, 'queue_time': 0.048028505, 'total_time': 0.090434587}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019f9432-8aec-70c0-b882-47d90a8b3e39-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 235, 'output_tokens': 25, 'total_tokens': 260})]}

As we can see it is REDACTED_EMAIL and also masked credit card